# Decentralized Autonomous Organizations (DAOs) -- Data gathering

**[Johnnatan Messias](https://johnnatan-messias.github.io/), January 2025**

This code gathers voting history of decentralized governance protocols.


In [1]:
import os
from web3 import Web3
import requests as re
import gzip
import pickle

In [2]:
import sys
code_dir = os.path.realpath(os.path.join(os.getcwd(), "..", "src"))

sys.path.append(code_dir)

In [3]:
from ethereum import to_checksum_address, get_contract, get_all_events_from_contract
from utils import load_contracts_info

In [4]:
# Set directory paths
data_dir = os.path.abspath(
    os.path.join(os.getcwd(), "..", "data")) + os.sep
plots_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "plots")) + os.sep

blocks_dir = os.path.abspath(os.path.join(data_dir, "blocks")) + os.sep
# Create directories if they don't exist
os.makedirs(data_dir, exist_ok=True)
os.makedirs(blocks_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

### Yahoo historical prices

In [ ]:
# Gathering Yahoo historical prices for ETH-USD
from datetime import date
import yfinance as yf
today = str(date.today())
price_date = yf.download('ETH-USD', end=today)

# Removing unnecessary Ticker level
price_date = price_date.droplevel('Ticker', axis=1)
price_date = price_date[['Open', 'High', 'Low', 'Close', 'Volume']]

# Persisting file
file_dir = os.path.abspath(os.path.join(data_dir, "yahoo_price_eth_usd.csv"))
price_date.to_csv(file_dir)
price_date.head()

### Loading Contracts and ABIs 

In [6]:
block_max_ethereum = 21_665_000
contract_settings = load_contracts_info(data_dir + 'contracts.json')

In [ ]:
# This code connects to Paradigm Reth archive node (see https://x.com/gakonst/status/1702389827390546071)
# Paradigm Reth archive node. It is a free service provided by Paradigm.
eth_node = 'http://69.67.151.138:8545'

adapter = re.adapters.HTTPAdapter(pool_connections=20, pool_maxsize=20)
session = re.Session()
session.mount('http://', adapter)
session.mount('https://', adapter)

w3 = Web3(Web3.HTTPProvider(eth_node, session=session,
          request_kwargs={'timeout': 60}))

print("Is connected to Ethereum node: ", w3.is_connected())
print("The most recent block is: ", w3.eth.block_number)

### Loading token contracts

In [8]:
def load_contracts(settings):
    contracts = {}
    for contract_setting in settings:
        print(f"Loading contract {contract_setting['name']} at address {
              contract_setting['address']}")
        contracts[contract_setting['name']] = get_contract(
            w3,
            to_checksum_address(contract_setting['address']),
            abi_contract_address=contract_setting['abi_address'], is_zksync=False)
    return contracts

In [ ]:
token_contracts = load_contracts(contract_settings['token'])
governance_contracts = load_contracts(contract_settings['governance'])

## Gathering Events


In [11]:
def gather_events(contracts, contract_settings, block_max_ethereum, batch_size=50, max_workers=10):
    # Gather all contract events from contract_settings
    for contract_setting in contract_settings:
        print('====== ', contract_setting['name'], ' ======')
        events = get_all_events_from_contract(contracts[contract_setting['name']],
                                              start_block=contract_setting['start'],
                                              end_block=block_max_ethereum,
                                              batch_size=batch_size,
                                              max_workers=max_workers,
                                              events=contract_setting['events'])
        file_dir = os.path.abspath(
            os.path.join(data_dir, f"events_{contract_setting['type']}_{contract_setting['name']}.pkl.gz"))
        with gzip.open(file_dir, 'wb') as f:
            pickle.dump(events, f)

In [ ]:
# Gatehring governance events
gather_events(governance_contracts,
              contract_settings['governance'], block_max_ethereum, batch_size=2500)

In [ ]:
# Gatehring token events
gather_events(token_contracts,
              contract_settings['token'], block_max_ethereum, batch_size=500)